# 06 — FT-Transformer models

FT-Transformer classifier training and evaluation (A2: autoencoder-reduced, B2: pooled features). Uses `src.models.ft_transformer` and `src.training.evaluation`.

In [1]:
# Path fix: use this repo's src
from pathlib import Path
import sys
import logging
import numpy as np
import torch
PROJECT_ROOT = Path.cwd().resolve().parents[0]
project_root_str = str(PROJECT_ROOT)
if project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)
for name in list(sys.modules.keys()):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

from src.utils import SEED, set_global_seed
from src.utils.paths import get_reduced_dir, get_splits_dir, get_experiment_dir, get_experiment_output_dir, ensure_dir
from src.data import CLASS_NAMES, load_labels
from src.data.splits import load_splits
from src.models.ft_transformer import FTTransformer
from src.training.evaluation import compute_metrics, compute_per_class_metrics, save_metrics, save_confusion_matrix

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

# FT-Transformer / training params (match A2 and B2 configs)
EPOCHS = 20
BATCH_SIZE = 256
LR = 1e-3
WEIGHT_DECAY = 1e-4
D_TOKEN = 128
N_HEADS = 8
N_LAYERS = 3
DROPOUT = 0.1
NUM_CLASSES = len(CLASS_NAMES)
print("FT-Transformer: epochs=%s batch=%s lr=%s | d_token=%s n_heads=%s n_layers=%s | %d classes: %s" % (EPOCHS, BATCH_SIZE, LR, D_TOKEN, N_HEADS, N_LAYERS, NUM_CLASSES, CLASS_NAMES))

FT-Transformer: epochs=20 batch=256 lr=0.001 | d_token=128 n_heads=8 n_layers=3 | 4 classes: ['AF', 'SVT', 'Sinus Brady', 'Sinus Rhythm']


In [2]:
# Load splits and labels (same order as saved train/test/val .npy)
print("Loading splits and labels...")
idx_train, idx_val, idx_test = load_splits()
try:
    y_full = load_labels()
except FileNotFoundError:
    print("labels.npy not found; loading from raw dataset...")
    from src.data import load_raw_dataset
    _, y_full = load_raw_dataset()
if y_full.ndim == 2:
    y_idx = np.argmax(y_full, axis=1)
else:
    y_idx = y_full
y_train = y_idx[idx_train]
y_test = y_idx[idx_test]
y_val = y_idx[idx_val] if len(idx_val) > 0 else None
print("y_train %s y_test %s" % (y_train.shape, y_test.shape))

Loading splits and labels...
y_train (36120,) y_test (6773,)


In [3]:
def train_ft_and_evaluate(X_train, X_val, X_test, y_train, y_val, y_test, n_features, condition_name):
    """Train FT-Transformer on train set; evaluate on validation (if present) then on test. Save checkpoint and metrics."""
    set_global_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    exp_dir = ensure_dir(get_experiment_dir(condition_name))
    ckpt_dir = ensure_dir(get_experiment_output_dir(condition_name, checkpoints=True))

    model = FTTransformer(
        n_features=n_features,
        num_classes=NUM_CLASSES,
        d_token=D_TOKEN,
        n_heads=N_HEADS,
        n_layers=N_LAYERS,
        dropout=DROPOUT,
    ).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    n_train = len(X_train)

    for ep in range(EPOCHS):
        model.train()
        perm = np.random.permutation(n_train)
        total_loss = 0.0
        n_b = 0
        for start in range(0, n_train, BATCH_SIZE):
            idx = perm[start : start + BATCH_SIZE]
            bx = torch.from_numpy(X_train[idx]).float().to(device)
            by = torch.from_numpy(y_train[idx]).long().to(device)
            logits = model(bx)
            loss = torch.nn.functional.cross_entropy(logits, by)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item()
            n_b += 1
        if (ep + 1) % 5 == 0 or ep == 0:
            print("  Epoch %d loss %.4f" % (ep + 1, total_loss / max(n_b, 1)))

    torch.save(model.state_dict(), ckpt_dir / "ft_transformer.pt")
    print("  Saved %s" % (ckpt_dir / "ft_transformer.pt"))

    model.eval()
    all_metrics = {}
    labels_list = list(range(NUM_CLASSES))
    with torch.no_grad():
        if X_val is not None and y_val is not None and len(y_val) > 0:
            logits_val = model(torch.from_numpy(X_val).float().to(device))
            y_pred_val = logits_val.argmax(dim=1).cpu().numpy()
            metrics_val = compute_metrics(y_val, y_pred_val, task="multiclass", labels=labels_list)
            all_metrics["weighted_f1_val"] = metrics_val["weighted_f1"]
            per_class_val = compute_per_class_metrics(y_val, y_pred_val, labels=labels_list, target_names=CLASS_NAMES)
            all_metrics["per_class_val"] = per_class_val
            print("  Validation weighted F1: %.4f" % metrics_val["weighted_f1"])
            print("  Validation per-class:")
            for p in per_class_val:
                print("    %s: P=%.3f R=%.3f F1=%.3f support=%d" % (p["class"], p["precision"], p["recall"], p["f1"], p["support"]))
        logits_test = model(torch.from_numpy(X_test).float().to(device))
        y_pred_test = logits_test.argmax(dim=1).cpu().numpy()
    metrics_test = compute_metrics(y_test, y_pred_test, task="multiclass", labels=labels_list)
    per_class_test = compute_per_class_metrics(y_test, y_pred_test, labels=labels_list, target_names=CLASS_NAMES)
    all_metrics["weighted_f1"] = metrics_test["weighted_f1"]
    all_metrics["confusion_matrix"] = metrics_test.get("confusion_matrix", [])
    all_metrics["per_class"] = per_class_test
    save_metrics(all_metrics, exp_dir / "metrics.json")
    cm = np.array(metrics_test["confusion_matrix"])
    save_confusion_matrix(cm, exp_dir / "confusion_matrix.npy", path_png=exp_dir / "confusion_matrix.png", class_names=CLASS_NAMES)
    print("  Test weighted F1: %.4f" % metrics_test["weighted_f1"])
    print("  Test per-class:")
    for p in per_class_test:
        print("    %s: P=%.3f R=%.3f F1=%.3f support=%d" % (p["class"], p["precision"], p["recall"], p["f1"], p["support"]))
    return all_metrics


def print_metrics_summary(metrics, model_name):
    """Print test metrics for one model (weighted F1 + per-class table)."""
    print("\n" + "=" * 60)
    print("  %s — Test metrics" % model_name)
    print("=" * 60)
    print("  Weighted F1: %.4f" % metrics.get("weighted_f1", 0))
    if "weighted_f1_val" in metrics:
        print("  Weighted F1 (val): %.4f" % metrics["weighted_f1_val"])
    per = metrics.get("per_class", [])
    if per:
        print("  Per-class: %-12s %8s %8s %8s %8s" % ("Class", "Precision", "Recall", "F1", "Support"))
        print("  " + "-" * 52)
        for p in per:
            print("  %-12s %8.3f %8.3f %8.3f %8d" % (p["class"], p["precision"], p["recall"], p["f1"], p["support"]))
    print("=" * 60)

In [ ]:
# A2: Autoencoder + FT-Transformer (load autoencoder-reduced features)
print("=== A2: Autoencoder + FT-Transformer ===")
red_dir = get_reduced_dir() / "autoencoder"
X_train_a2 = np.load(red_dir / "train.npy").astype(np.float32)
X_test_a2 = np.load(red_dir / "test.npy").astype(np.float32)
if (red_dir / "val.npy").exists():
    X_val_a2 = np.load(red_dir / "val.npy").astype(np.float32)
else:
    X_val_a2 = None
n_features_a2 = X_train_a2.shape[1]
print("Loaded autoencoder features: train %s test %s (n_features=%d)" % (X_train_a2.shape, X_test_a2.shape, n_features_a2))
metrics_a2 = train_ft_and_evaluate(X_train_a2, X_val_a2, X_test_a2, y_train, y_val, y_test, n_features_a2, "A2_autoencoder_ft")
print_metrics_summary(metrics_a2, "A2 (Autoencoder + FT-Transformer)")

=== A2: Autoencoder + FT-Transformer ===
Loaded autoencoder features: train (36120, 256) test (6773, 256) (n_features=256)


2026-02-28 13:53:52,933 | INFO | Global seed set to 0


In [ ]:
# B2: Pooling + FT-Transformer (load pooled features)
print("=== B2: Pooling + FT-Transformer ===")
red_dir = get_reduced_dir() / "pooled"
X_train_b2 = np.load(red_dir / "train.npy").astype(np.float32)
X_test_b2 = np.load(red_dir / "test.npy").astype(np.float32)
if (red_dir / "val.npy").exists():
    X_val_b2 = np.load(red_dir / "val.npy").astype(np.float32)
else:
    X_val_b2 = None
n_features_b2 = X_train_b2.shape[1]
print("Loaded pooled features: train %s test %s (n_features=%d)" % (X_train_b2.shape, X_test_b2.shape, n_features_b2))
metrics_b2 = train_ft_and_evaluate(X_train_b2, X_val_b2, X_test_b2, y_train, y_val, y_test, n_features_b2, "B2_pooling_ft")
print_metrics_summary(metrics_b2, "B2 (Pooling + FT-Transformer)")

In [ ]:
# Summary: overall and per-class (test set)
print("=== Summary ===")
print("A2 (autoencoder + FT-Transformer) test weighted F1: %.4f" % metrics_a2["weighted_f1"])
print("B2 (pooling + FT-Transformer) test weighted F1: %.4f" % metrics_b2["weighted_f1"])
print("\nPer-class (test set):")
print("%-12s | %8s %8s %8s | %8s %8s %8s" % ("Class", "A2_P", "A2_R", "A2_F1", "B2_P", "B2_R", "B2_F1"))
print("-" * 70)
for i, name in enumerate(CLASS_NAMES):
    a2 = metrics_a2.get("per_class", [])[i] if "per_class" in metrics_a2 else {}
    b2 = metrics_b2.get("per_class", [])[i] if "per_class" in metrics_b2 else {}
    print("%-12s | %8.3f %8.3f %8.3f | %8.3f %8.3f %8.3f" % (
        name,
        a2.get("precision", 0), a2.get("recall", 0), a2.get("f1", 0),
        b2.get("precision", 0), b2.get("recall", 0), b2.get("f1", 0),
    ))